# Notebook 1: Inspecting and Cleaning the Questionnaire Data

**Project:** AI vs. Human Translation: A Comparative Study Based on Student Perceptions (MA thesis, University of Oran 2)

In this notebook I take the raw Google Forms export and prepare it for analysis. The steps are:

1. Import the data and look at its size, columns and data types
2. Check missing values, duplicates and inconsistent answers
3. Rename the columns
4. Clean the categorical variables and convert data types
5. Filter rows and create subsets
6. Handle missing data
7. Export the cleaned dataset (CSV and Excel)

**Why this matters in linguistic research:** questionnaire data from online forms is almost never clean.
Typos, extra spaces and free-text answers to yes/no questions can silently change frequency counts, so cleaning has to come first.

## 1. Import the libraries and the data

In [1]:
import pandas as pd   # tables of data (DataFrames)
import numpy as np     # numerical functions

In [2]:
# read_csv() loads a CSV file into a DataFrame (a table with rows and columns)
df_raw = pd.read_csv("../data/raw/survey_responses_raw.csv")

# I keep the untouched copy in df_raw and work on a copy called df
df = df_raw.copy()
print("Rows and columns:", df.shape)

Rows and columns: (32, 20)


## 2. First inspection

In [3]:
# head() shows the first rows. I only look at the first 3 columns here to keep the output short.
print(df.iloc[:3, :4])

            Timestamp         Age: Gender: Educational level?
0  2/16/2025 22:11:38  20-27 years  Female             Master
1  2/16/2025 23:02:12  20-27 years  Female             Master
2  2/16/2025 23:03:33  20-27 years  Female             Master


In [4]:
# The column names are the full questions from the form: very long and hard to type
for number, name in enumerate(df.columns):
    print(number, repr(name))

0 'Timestamp'
1 'Age:'
2 'Gender:'
3 'Educational level?'
4 "What's your field of study?"
5 "What's your profession ? "
6 "What's your language proficiency level?"
7 'Have you ever worked as a professional translator or received formal translation training?'
8 'Have you ever used AI-powered translation tools (e.g., Google Translate,DeepL, ChatGPT, etc.)?'
9 'How often do you use Ai translation ?'
10 'Have you ever used human translation services (e.g., professional translators, freelancers, or agencies)?'
11 ' Do you prefer AI translation or human translation? Why?  '
12 'How would you rate the accuracy of AI translation compared to human translation on a scale of 1-10?  '
13 'Which method do you find faster, and does speed influence your preference?'
14 'How well do you think each method handles context and idiomatic expressions?'
15 'How satisfied are you with the results of AI translation versus human translation?  '
16 'Are you willing to pay more for human translation, and why?  '

In [5]:
# dtypes tells me the type of each column (text, whole numbers, decimals...)
print(df.dtypes)

Timestamp                                                                                                        str
Age:                                                                                                             str
Gender:                                                                                                          str
Educational level?                                                                                               str
What's your field of study?                                                                                      str
What's your profession ?                                                                                         str
What's your language proficiency level?                                                                          str
Have you ever worked as a professional translator or received formal translation training?                       str
Have you ever used AI-powered translation tools (e.g., Google Tr

Only the two rating questions were read as numbers (`int64`). Everything else is text.
The last column (`Age:.1`) is read as `float64`, which is suspicious: it may be empty.

### 2.1 Missing values and duplicates

In [6]:
# isna() marks missing cells as True, sum() counts them per column
print(df.isna().sum())

Timestamp                                                                                                     0
Age:                                                                                                          0
Gender:                                                                                                       0
Educational level?                                                                                            0
What's your field of study?                                                                                   0
What's your profession ?                                                                                      9
What's your language proficiency level?                                                                       0
Have you ever worked as a professional translator or received formal translation training?                    2
Have you ever used AI-powered translation tools (e.g., Google Translate,DeepL, ChatGPT, etc.)?          

In [7]:
# Is the last column completely empty?
print("Empty cells in 'Age:.1':", df["Age:.1"].isna().sum(), "out of", len(df))

# Are there duplicated rows (the same person submitting twice)?
print("Duplicated rows:", df.duplicated().sum())

Empty cells in 'Age:.1': 32 out of 32
Duplicated rows: 0


- `Age:.1` is empty for all 32 rows. It is an accidental duplicate of the age question, so I will delete it.
- There are no duplicated rows.
- Several open-ended questions have missing values. This is probably because those questions were not compulsory.

### 2.2 Unique values in the categorical questions

In [8]:
# value_counts() counts how many times each answer appears.
# dropna=False also shows the missing values.
questions_to_check = ["Age:", "Gender:", "Educational level?", "What's your field of study?",
                      "What's your language proficiency level?"]

for question in questions_to_check:
    print(question)
    print(df[question].value_counts(dropna=False))
    print()

Age:
Age:
20-27 years       31
Under 20 years     1
Name: count, dtype: int64

Gender:
Gender:
Female    28
Male       4
Name: count, dtype: int64

Educational level?
Educational level?
Master     28
Licence     3
PhD         1
Name: count, dtype: int64

What's your field of study?
What's your field of study?
Linguistics and Applied languages    24
others                                5
Literature and civilization           3
Name: count, dtype: int64

What's your language proficiency level?
What's your language proficiency level?
C1-C2    17
B1-B2    15
Name: count, dtype: int64



In [9]:
# Now the yes/no questions. Here I expect only 'Yes' and 'No'.
yes_no_questions = [
    "Have you ever used AI-powered translation tools (e.g., Google Translate,DeepL, ChatGPT, etc.)?",
    "How often do you use Ai translation ?",
    "Have you ever used human translation services (e.g., professional translators, freelancers, or agencies)?",
]
for question in yes_no_questions:
    print(df[question].value_counts(dropna=False))
    print()

Have you ever used AI-powered translation tools (e.g., Google Translate,DeepL, ChatGPT, etc.)?
Yes                31
Not very often      1
Name: count, dtype: int64

How often do you use Ai translation ?
Somtimes    26
Always       5
NaN          1
Name: count, dtype: int64

Have you ever used human translation services (e.g., professional translators, freelancers, or agencies)?
No     21
Yes    10
NaN     1
Name: count, dtype: int64



**Problems found so far**

- One student answered **"Not very often"** to a Yes/No question ("Have you ever used AI translation tools?").
- "Somtimes" is a typo for "Sometimes".
- "others" starts with a lowercase letter.

In [10]:
# The translation-training question was open text, so let me look at all answers
training_question = "Have you ever worked as a professional translator or received formal translation training?"
print(df[training_question].value_counts(dropna=False))

Have you ever worked as a professional translator or received formal translation training?
No                                                                                                                                               21
NaN                                                                                                                                               2
No                                                                                                                                                2
never                                                                                                                                             1
Yes                                                                                                                                               1
No, i haven't                                                                                                                                     1
I have received a tra

This question should be Yes/No but it has 9 different spellings ("No ", "No", "never", "Never did", "no i have not"...).
Some answers also contain a trailing space, so `"No "` and `"No"` are counted as different values.
I will fix this step by step in the next section.

## 3. Rename the columns

In [11]:
# I replace the long questions by short names without spaces.
# The new names must be in the same order as the old ones.
new_names = [
    "timestamp", "age_group", "gender", "edu_level", "field", "profession",
    "proficiency", "translation_training", "used_ai", "ai_frequency",
    "used_human", "preference_text", "ai_accuracy_rating", "faster_text",
    "context_idiom_text", "satisfaction", "pay_more_text",
    "improvements_text", "future_text", "empty_column",
]
df.columns = new_names
print(list(df.columns))

['timestamp', 'age_group', 'gender', 'edu_level', 'field', 'profession', 'proficiency', 'translation_training', 'used_ai', 'ai_frequency', 'used_human', 'preference_text', 'ai_accuracy_rating', 'faster_text', 'context_idiom_text', 'satisfaction', 'pay_more_text', 'improvements_text', 'future_text', 'empty_column']


In [12]:
# Remove the empty column. axis=1 means "a column" (axis=0 would be a row).
df = df.drop("empty_column", axis=1)

# Add a participant ID (P01 ... P32) so I can always trace a row back to the raw file.
# zfill(2) adds a leading zero: "1" becomes "01".
df["participant_id"] = ["P" + str(number).zfill(2) for number in range(1, len(df) + 1)]
print(df["participant_id"].head())
print("Columns now:", df.shape[1])

0    P01
1    P02
2    P03
3    P04
4    P05
Name: participant_id, dtype: str
Columns now: 20


## 4. Clean the variables

### 4.1 Remove extra spaces and fix spelling

In [13]:
# str.strip() removes spaces at the start and end of each text answer
text_columns = ["age_group", "gender", "edu_level", "field", "profession", "proficiency",
                "translation_training", "used_ai", "ai_frequency", "used_human", "preference_text",
                "faster_text", "context_idiom_text", "pay_more_text", "improvements_text", "future_text"]

for column in text_columns:
    df[column] = df[column].str.strip()

# replace() changes specific values
df["ai_frequency"] = df["ai_frequency"].replace("Somtimes", "Sometimes")
df["field"] = df["field"].replace("others", "Other")

print(df["ai_frequency"].value_counts(dropna=False))
print(df["field"].value_counts())

ai_frequency
Sometimes    26
Always        5
NaN           1
Name: count, dtype: int64
field
Linguistics and Applied languages    24
Other                                 5
Literature and civilization           3
Name: count, dtype: int64


### 4.2 The inconsistent answer in `used_ai`

In [14]:
# Which student gave the inconsistent answer?
print(df[df["used_ai"] == "Not very often"][["participant_id", "used_ai", "ai_frequency"]])

  participant_id         used_ai ai_frequency
0            P01  Not very often          NaN


This student did not answer the frequency question either. Because the answer is not a clean Yes/No, I do **not** change it.
The dissertation (Table 3.7) also keeps it as its own category, "Not Very Often" (1 student), so keeping it lets me reproduce the thesis numbers.
I only add a flag column so that this student is easy to find later.

In [15]:
# np.where(condition, value_if_true, value_if_false) is like an if/else for a whole column
df["flag_used_ai_inconsistent"] = np.where(df["used_ai"] == "Not very often", "Yes", "No")

print(df["used_ai"].value_counts())
print(df["flag_used_ai_inconsistent"].value_counts())

used_ai
Yes               31
Not very often     1
Name: count, dtype: int64
flag_used_ai_inconsistent
No     31
Yes     1
Name: count, dtype: int64


### 4.3 Recode the translation-training question

In [16]:
# I write a small function: it takes one answer and returns "Yes", "No" or "Not answered".
def code_training(answer):
    if pd.isna(answer):               # missing answer
        return "Not answered"
    answer = answer.lower()           # lower case, so "Yes" and "yes" are treated the same
    if answer.startswith("yes"):      # "Yes", "Yes recently"
        return "Yes"
    if "received a training" in answer:   # the Mediterranean Games volunteer training
        return "Yes"
    return "No"

# apply() runs the function on every value of the column
df["translation_training"] = df["translation_training"].apply(code_training)
print(df["translation_training"].value_counts())

translation_training
No              27
Yes              3
Not answered     2
Name: count, dtype: int64


*Methodological notes:*

- I counted the volunteer training at the Mediterranean Games as "Yes" (a judgement call that I record here).
- **To check:** the dissertation's Table 3.6 reports 26 "No / Never" and 6 "Yes / Conditional" (N = 32). My coding gives 27 No, 3 Yes and 2 not answered.
  I could not reconstruct how the thesis grouped the borderline and missing answers, so this difference has to be checked against the thesis coding before I quote either set of numbers.

### 4.4 Data types and ordered categories

In [17]:
# Dates: the timestamp is text, so I convert it to a real date/time
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %H:%M:%S")
print(df["timestamp"].min(), "->", df["timestamp"].max())

2025-02-16 22:11:38 -> 2025-02-18 11:33:53


In [18]:
# Frequency is an ordered variable: Sometimes < Always. A number version is useful for calculations.
# map() replaces each value using a dictionary. Missing values stay missing (NaN).
df["ai_frequency_num"] = df["ai_frequency"].map({"Sometimes": 1, "Always": 2})
print(df[["ai_frequency", "ai_frequency_num"]].drop_duplicates())

  ai_frequency  ai_frequency_num
0          NaN               NaN
1    Sometimes               1.0
5       Always               2.0


In [19]:
# Profession is free text with 20 different answers. I group it into 4 simple categories.
def code_profession(answer):
    if pd.isna(answer):
        return "Not stated"
    answer = answer.lower()
    if "student" in answer:
        return "Student"
    if "teacher" in answer:
        return "Teacher"
    return "Other"

df["profession_group"] = df["profession"].apply(code_profession)
print(df["profession_group"].value_counts())

profession_group
Student       9
Not stated    9
Other         7
Teacher       7
Name: count, dtype: int64


In [20]:
# Check that the ratings are inside the possible range
# (accuracy 1 to 10, satisfaction 1 to 5). between() returns True/False for each row.
print("All accuracy ratings between 1 and 10:", df["ai_accuracy_rating"].between(1, 10).all())
print("All satisfaction ratings between 1 and 5:", df["satisfaction"].between(1, 5).all())

All accuracy ratings between 1 and 10: True
All satisfaction ratings between 1 and 5: True


## 5. Filtering and creating subsets

In [21]:
# A boolean mask is a True/False column. Putting it inside df[...] keeps only the True rows.
students_who_use_ai = df[df["used_ai"] == "Yes"]
print("Students who used AI translation:", len(students_who_use_ai))

master_students = df[df["edu_level"] == "Master"]
print("Master students:", len(master_students))

# Two conditions: use & (and) and put each condition in parentheses
advanced_and_linguistics = df[(df["proficiency"] == "C1-C2") & (df["field"] == "Linguistics and Applied languages")]
print("C1-C2 students in Linguistics and Applied languages:", len(advanced_and_linguistics))

Students who used AI translation: 31
Master students: 28
C1-C2 students in Linguistics and Applied languages: 11


In [22]:
# Selecting only some columns
core_columns = df[["participant_id", "age_group", "gender", "edu_level", "field", "proficiency"]]
print(core_columns.head())

  participant_id    age_group  ...                              field proficiency
0            P01  20-27 years  ...        Literature and civilization       C1-C2
1            P02  20-27 years  ...  Linguistics and Applied languages       B1-B2
2            P03  20-27 years  ...  Linguistics and Applied languages       B1-B2
3            P04  20-27 years  ...  Linguistics and Applied languages       C1-C2
4            P05  20-27 years  ...  Linguistics and Applied languages       C1-C2

[5 rows x 6 columns]


## 6. Handling missing data

I do **not** delete students with missing answers: with only 32 participants every row matters, and a missing answer on
one open-ended question does not make the other answers unusable. Instead:

- Missing text answers are kept as missing (`NaN`) and each analysis only uses the students who answered that question.
- For yes/no questions, I replace missing values by "Not answered" so they remain visible in the tables.
- I do not invent a value for the missing AI-frequency answer (that would be guessing).

In [23]:
# fillna() fills the empty cells with a value of my choice
df["used_human"] = df["used_human"].fillna("Not answered")
print(df["used_human"].value_counts())
print()
print("Missing AI frequency:", df["ai_frequency"].isna().sum())

used_human
No              21
Yes             10
Not answered     1
Name: count, dtype: int64

Missing AI frequency: 1


In [24]:
# Subset for questions where an answer is required, e.g. only students who wrote a preference
students_with_preference = df[df["preference_text"].notna()]
print("Students who answered the preference question:", len(students_with_preference), "of", len(df))

Students who answered the preference question: 31 of 32


## 7. Export the cleaned data

In [25]:
# index=False prevents pandas from writing an extra column with row numbers
df.to_csv("../data/processed/survey_notebook_clean.csv", index=False)
df.to_excel("../data/processed/survey_notebook_clean.xlsx", index=False)

# Importing the Excel file back to check that it worked (read_excel is the Excel version of read_csv)
check = pd.read_excel("../data/processed/survey_notebook_clean.xlsx")
print("Rows and columns in the exported file:", check.shape)

Rows and columns in the exported file: (32, 23)


## Summary of cleaning

| Problem | What I did |
|---|---|
| Very long column names | Renamed with short names |
| Empty duplicate column | Dropped |
| Extra spaces, "Somtimes", "others" | Fixed with `strip()` and `replace()` |
| "Not very often" in a Yes/No question | Kept as its own category (as in Table 3.7) and flagged |
| 9 spellings of Yes/No in the training question | Recoded with a small function |
| Messy profession text | Grouped into 4 categories |
| Missing values | Kept, and made visible instead of deleted |

Next: **Notebook 2** creates the coded variables I need for the analysis.